In [3]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ================================================

import numpy as np
from scipy import ndimage
from skimage.morphology import skeletonize
from sklearn.cluster import DBSCAN
import json

W, H, res = 120, 80, 0.1

# ============================================================
# Step 0
# ============================================================
def make_grid():
    g = np.ones((H, W), dtype=np.uint8) * 255
    def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
    def c(cy, cx, r):
        Y, X = np.ogrid[:H, :W]
        g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0
    w(0, 3, 0, W); w(H-3, H, 0, W); w(0, H, 0, 3); w(0, H, W-3, W)
    w(3, 30, 55, 58); w(42, 77, 55, 58)
    w(3, 32, 80, 83); w(48, 77, 80, 83)
    w(95, 98, 90, 115); w(95, 74, 90, 93)
    c(20, 25, 2); c(20, 35, 2); c(28, 30, 2); c(28, 40, 2)
    c(65, 12, 3)
    w(8, 22, 95, 112)
    return g

grid = make_grid()

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(grid, cmap='gray', origin='upper')
ax.set_title("Step 0: Raw SLAM Grid Map\n(Black=Wall/Obstacle, White=Free)", fontsize=13)
ax.text(28, 40, "Living Rm", ha='center', fontsize=11, color='blue', weight='bold')
ax.text(68, 40, "Corridor", ha='center', fontsize=9, color='blue', weight='bold')
ax.text(100, 24, "Bedroom", ha='center', fontsize=11, color='blue', weight='bold')
ax.text(100, 63, "Kitchen", ha='center', fontsize=11, color='blue', weight='bold')
ax.annotate("Door1", xy=(56, 36), fontsize=10, color='red', weight='bold',
            xytext=(46, 50), arrowprops=dict(arrowstyle='->', color='red'))
ax.annotate("Door2", xy=(81, 40), fontsize=10, color='red', weight='bold',
            xytext=(73, 52), arrowprops=dict(arrowstyle='->', color='red'))
plt.tight_layout()
plt.savefig('step0_grid.png', dpi=150, bbox_inches='tight')
plt.close()
print("Step 0 done")

# ============================================================
# Step 1: Distance Transform
# ============================================================
free = (grid == 255).astype(np.uint8)
dist = ndimage.distance_transform_edt(free)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(dist, cmap='hot', origin='upper')
plt.colorbar(im, ax=ax, label='Distance to nearest obstacle (px)')
ax.set_title("Step 1: Distance Transform\n(Brighter = farther from wall = safer)", fontsize=13)
plt.tight_layout()
plt.savefig('step1_dist.png', dpi=150, bbox_inches='tight')
plt.close()
print("Step 1 done")

# ============================================================
# Step 2: Safe threshold + Voronoi Skeleton
# ============================================================
threshold_px = 5
safe = (dist >= threshold_px).astype(np.uint8) * 255
skeleton = skeletonize(safe > 0).astype(np.uint8) * 255

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(safe, cmap='gray', origin='upper')
axes[0].set_title(f"Step 2a: Safe Zone (>= {threshold_px*res}m from wall)", fontsize=12)
axes[1].imshow(skeleton, cmap='gray', origin='upper')
axes[1].set_title("Step 2b: Voronoi Skeleton\n(Centerline = safest path)", fontsize=12)
plt.tight_layout()
plt.savefig('step2_skeleton.png', dpi=150, bbox_inches='tight')
plt.close()
print("Step 2 done")

# ============================================================
# Step 3: DBSCAN Clustering
# ============================================================
sy, sx = np.where(skeleton > 0)
points = np.column_stack([sx, sy])
print(f"Skeleton points: {len(points)}")

# eps_px = 25
eps_px = 15  # 减小邻域半径
min_samples = 10  # 增加最小样本数
db = DBSCAN(eps=eps_px, min_samples=min_samples).fit(points)
labels = db.labels_
unique_labels = sorted(set(labels) - {-1})
print(f"Clusters: {len(unique_labels)}  (label=-1 is noise/doorway)")

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Set2(np.linspace(0, 1, max(len(unique_labels), 1)))
for i, lab in enumerate(unique_labels):
    mask = labels == lab
    ax.scatter(points[mask, 0], points[mask, 1], s=1,
               color=colors[i % len(colors)], label=f'Room_{lab}')
noise = labels == -1
if noise.any():
    ax.scatter(points[noise, 0], points[noise, 1], s=1,
               color='gray', alpha=0.3, label='noise (doorway)')
ax.imshow(grid, cmap='gray_r', origin='upper', alpha=0.3)
ax.legend(markerscale=10, loc='upper right')
ax.set_title("Step 3: DBSCAN Clustering\n(Each color = one topological node/room)", fontsize=13)
plt.tight_layout()
plt.savefig('step3_cluster.png', dpi=150, bbox_inches='tight')
plt.close()
print("Step 3 done")

# ============================================================
# Step 4: Topology Extraction
# ============================================================
nodes = []
for lab in unique_labels:
    mask = labels == lab
    cx = np.mean(points[mask, 0]) * res
    cy = np.mean(points[mask, 1]) * res
    nodes.append({"id": f"Room_{lab}", "x": round(cx, 2), "y": round(cy, 2),
                  "skeleton_points": int(mask.sum())})

noise_pts = points[labels == -1]
edges = set()
for i in range(len(unique_labels)):
    for j in range(i + 1, len(unique_labels)):
        pts_i = points[labels == unique_labels[i]]
        pts_j = points[labels == unique_labels[j]]
        mid = (pts_i.mean(axis=0) + pts_j.mean(axis=0)) / 2
        if len(noise_pts) > 0:
            dists = np.sqrt(((noise_pts - mid)**2).sum(axis=1))
            if dists.min() < eps_px * 2:
                edges.add((unique_labels[i], unique_labels[j]))

edges_list = [{"from": f"Room_{e[0]}", "to": f"Room_{e[1]}"} for e in edges]
print(f"Edges found: {len(edges_list)}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(grid, cmap='gray_r', origin='upper', alpha=0.4)
ax.scatter(points[:, 0], points[:, 1], s=0.5, color='lightblue', alpha=0.5)
if len(noise_pts) > 0:
    ax.scatter(noise_pts[:, 0], noise_pts[:, 1], s=0.5, color='yellow', alpha=0.5)
for n in nodes:
    px, py = n['x'] / res, n['y'] / res
    ax.plot(px, py, 'o', markersize=14, color='red', markeredgecolor='black', markeredgewidth=2)
    ax.annotate(f"{n['id']}\n({n['x']}, {n['y']})", (px, py),
                textcoords="offset points", xytext=(10, 10), fontsize=9, color='red', weight='bold')
for e in edges_list:
    n1 = next(n for n in nodes if n['id'] == e['from'])
    n2 = next(n for n in nodes if n['id'] == e['to'])
    ax.plot([n1['x']/res, n2['x']/res], [n1['y']/res, n2['y']/res],
            'g--', linewidth=2, alpha=0.8)
    mx = (n1['x'] + n2['x']) / 2 / res
    my = (n1['y'] + n2['y']) / 2 / res
    ax.annotate("Door", (mx, my), fontsize=10, color='green', weight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
ax.set_title("Step 4: Final Topology Map\n(Red=Room center, Green dash=Door, Blue=Skeleton)", fontsize=13)
plt.tight_layout()
plt.savefig('step4_topology.png', dpi=150, bbox_inches='tight')
plt.close()
print("Step 4 done")

topology = {"nodes": nodes, "edges": edges_list}
print("\n" + "=" * 50)
print("JSON for LLM:")
print("=" * 50)
print(json.dumps(topology, indent=2, ensure_ascii=False))


Step 0 done
Step 1 done
Step 2 done
Skeleton points: 319
Clusters: 1  (label=-1 is noise/doorway)
Step 3 done
Edges found: 0
Step 4 done

JSON for LLM:
{
  "nodes": [
    {
      "id": "Room_0",
      "x": 5.25,
      "y": 3.62,
      "skeleton_points": 319
    }
  ],
  "edges": []
}


In [28]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.morphology import skeletonize
from sklearn.cluster import DBSCAN
import json

W, H, res = 120, 80, 0.1

def make_grid():
    g = np.ones((H, W), dtype=np.uint8) * 255
    def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
    def c(cy, cx, r):
        Y, X = np.ogrid[:H, :W]
        g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0
    w(0, 3, 0, W); w(H-3, H, 0, W); w(0, H, 0, 3); w(0, H, W-3, W)
    w(3, 30, 55, 58); w(42, 77, 55, 58)
    w(3, 32, 80, 83); w(48, 77, 80, 83)
    w(95, 98, 90, 115); w(95, 74, 90, 93)
    c(20, 25, 2); c(20, 35, 2); c(28, 30, 2); c(28, 40, 2)
    c(65, 12, 3)
    w(8, 22, 95, 112)
    return g

grid = make_grid()

# Step 1: Distance Transform
free = (grid == 255).astype(np.uint8)
dist = ndimage.distance_transform_edt(free)

# Step 2: Safe threshold + Voronoi Skeleton
threshold_px = 3
safe = (dist >= threshold_px).astype(np.uint8) * 255
skeleton = skeletonize(safe > 0).astype(np.uint8) * 255

# 调试：打印骨架图统计信息
print(f"骨架图非零像素数: {np.sum(skeleton)}")
print(f"骨架图形状: {skeleton.shape}")

# Step 3: 从骨架图中提取分叉点（junction points）
skeleton_uint8 = skeleton.astype(np.uint8)
neighbor_count = ndimage.convolve(skeleton_uint8, np.ones((3, 3)), mode='constant', cval=0)
junction_points = (neighbor_count >= 3) & skeleton_uint8

# 获取分叉点坐标
junction_y, junction_x = np.where(junction_points)
junction_coords = np.column_stack([junction_x, junction_y])

print(f"分叉点数量: {len(junction_coords)}")

# # 可视化分叉点
# fig, ax = plt.subplots(figsize=(10, 6))
# ax.imshow(grid, cmap='gray', origin='upper', alpha=0.4)
# ax.scatter(junction_x, junction_y, s=30, color='red', marker='o', label='Junction Points')
# ax.set_title("Step 3: Junction Points (Red Dots)", fontsize=13)
# plt.tight_layout()
# plt.savefig('step3_junction_points.png', dpi=150, bbox_inches='tight')
# plt.close()
# print("Step 3 done")
# Step 3: 从骨架图中提取分叉点（junction points）—— 修正版
skeleton_bin = (skeleton > 0).astype(np.uint8)          # 0/1 二值图
neighbor_count = ndimage.convolve(skeleton_bin, np.ones((3, 3)), mode='constant', cval=0)
junction_points = (neighbor_count >= 4) & skeleton_bin  # 正确阈值

junction_y, junction_x = np.where(junction_points)
junction_coords = np.column_stack([junction_x, junction_y])

print(f"修正后分叉点数量: {len(junction_coords)}")
# Step 4: 对分叉点使用DBSCAN聚类
if len(junction_coords) > 0:
    # 调整DBSCAN参数：eps=10（更小的邻域半径），min_samples=5（更多样本）
    db = DBSCAN(eps=3, min_samples=3).fit(junction_coords)
    labels = db.labels_
    unique_labels = sorted(set(labels) - {-1})  # 排除噪声点（label=-1）
    print(f"聚类出 {len(unique_labels)} 个区域 (label=-1是噪声,忽略)")
    
    # 打印每个聚类的点数
    for label in unique_labels:
        cluster_points = junction_coords[labels == label]
        print(f"Cluster {label}: {len(cluster_points)} points")
    
    # 计算每个聚类的中心作为节点
    nodes = []
    for label in unique_labels:
        cluster_points = junction_coords[labels == label]
        cx = np.mean(cluster_points[:, 0]) * res
        cy = np.mean(cluster_points[:, 1]) * res
        nodes.append({
            "id": f"Cluster_{label}",
            "x": round(cx, 2),
            "y": round(cy, 2),
            "points": len(cluster_points)
        })
    
    # 寻找边：连接相邻的聚类中心
    edges = []
    if len(nodes) > 1:
        # 计算节点之间的距离
        node_coords = np.array([[n['x']/res, n['y']/res] for n in nodes])
        dist_matrix = np.linalg.norm(node_coords[:, np.newaxis] - node_coords, axis=2)
        
        # 设置距离阈值（例如 30 像素 = 3米）
        threshold_dist = 30
        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                if dist_matrix[i, j] < threshold_dist:
                    edges.append((i, j))
    
    edges_list = [{"from": f"Cluster_{e[0]}", "to": f"Cluster_{e[1]}"} for e in edges]
    print(f"Edges found: {len(edges_list)}")
    print(f"Nodes: {nodes}")

# 最终可视化
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(grid, cmap='gray', origin='upper', alpha=0.4)
# 画骨架
ax.scatter(np.where(skeleton)[1], np.where(skeleton)[0], s=0.5, color='lightblue', alpha=0.5)
# 画节点
for n in nodes:
    px, py = n['x'] / res, n['y'] / res
    ax.plot(px, py, 'o', markersize=14, color='red', markeredgecolor='black', markeredgewidth=2)
    ax.annotate(f"{n['id']}\n({n['x']}, {n['y']})", (px, py),
                textcoords="offset points", xytext=(10, 10), fontsize=9, color='red', weight='bold')
# 画边
for e in edges_list:
    n1 = nodes[e[0]]
    n2 = nodes[e[1]]
    ax.plot([n1['x']/res, n2['x']/res], [n1['y']/res, n2['y']/res],
            'g--', linewidth=2, alpha=0.8)
ax.set_title("Step 4: Final Topology Map\n(Red=Cluster Center, Green dash=Connection)", fontsize=13)
plt.tight_layout()
plt.savefig('step4_topology.png', dpi=150, bbox_inches='tight')
plt.close()
print("Step 4 done")

topology = {"nodes": nodes, "edges": edges_list}
print("\n" + "=" * 50)
print("JSON for LLM:")
print("=" * 50)
print(json.dumps(topology, indent=2, ensure_ascii=False))


骨架图非零像素数: 112965
骨架图形状: (80, 120)
分叉点数量: 443
修正后分叉点数量: 33
聚类出 7 个区域 (label=-1是噪声,忽略)
Cluster 0: 4 points
Cluster 1: 4 points
Cluster 2: 4 points
Cluster 3: 8 points
Cluster 4: 4 points
Cluster 5: 3 points
Cluster 6: 6 points
Edges found: 6
Nodes: [{'id': 'Cluster_0', 'x': np.float64(3.0), 'y': np.float64(1.03), 'points': 4}, {'id': 'Cluster_1', 'x': np.float64(1.5), 'y': np.float64(3.42), 'points': 4}, {'id': 'Cluster_2', 'x': np.float64(4.82), 'y': np.float64(3.5), 'points': 4}, {'id': 'Cluster_3', 'x': np.float64(6.8), 'y': np.float64(3.7), 'points': 8}, {'id': 'Cluster_4', 'x': np.float64(9.88), 'y': np.float64(3.9), 'points': 4}, {'id': 'Cluster_5', 'x': np.float64(1.73), 'y': np.float64(4.57), 'points': 3}, {'id': 'Cluster_6', 'x': np.float64(3.5), 'y': np.float64(4.85), 'points': 6}]


KeyError: 0

In [7]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.morphology import skeletonize
from sklearn.cluster import DBSCAN
import json

W, H, res = 120, 80, 0.1

def make_grid():
    g = np.ones((H, W), dtype=np.uint8) * 255
    def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
    def c(cy, cx, r):
        Y, X = np.ogrid[:H, :W]
        g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0
    w(0, 3, 0, W); w(H-3, H, 0, W); w(0, H, 0, 3); w(0, H, W-3, W)
    w(3, 30, 55, 58); w(42, 77, 55, 58)
    w(3, 32, 80, 83); w(48, 77, 80, 83)
    w(95, 98, 90, 115); w(74, 95, 90, 93) # 修正了这里：原本是 w(95, 74, 90, 93) 导致这堵墙没画上
    c(20, 25, 2); c(20, 35, 2); c(28, 30, 2); c(28, 40, 2)
    c(65, 12, 3)
    w(8, 22, 95, 112)
    return g

grid = make_grid()

# ==========================================
# Step 1: 距离变换与 Safe 阈值分割
# ==========================================
free = (grid == 255).astype(np.uint8)
dist = ndimage.distance_transform_edt(free)

# 加大阈值到 5 (0.5米)，确保墙壁附近的走廊被完全掐断，把房间彻底分开
threshold_px = 5 
safe = (dist >= threshold_px).astype(np.uint8) * 255

# ==========================================
# Step 2: 骨架化与分叉点提取
# ==========================================
skeleton = skeletonize(safe > 0).astype(np.uint8)

kernel = np.array([[1, 1, 1],
                   [1, 0, 1],
                   [1, 1, 1]])
neighbor_count = ndimage.convolve(skeleton, kernel, mode='constant', cval=0)
junction_mask = (neighbor_count >= 3) & (skeleton > 0)
jy, jx = np.where(junction_mask)
junction_coords = np.column_stack([jx, jy]) if len(jx) > 0 else np.empty((0, 2))

# ==========================================
# Step 3: DBSCAN 聚类分叉点 -> 生成“路口节点”
# ==========================================
junction_nodes = []
if len(junction_coords) > 0:
    db = DBSCAN(eps=12, min_samples=2).fit(junction_coords)
    for label in sorted(set(db.labels_) - {-1}):
        pts = junction_coords[db.labels_ == label]
        cx = np.mean(pts[:, 0]) * res
        cy = np.mean(pts[:, 1]) * res
        junction_nodes.append({
            "id": f"Junc_{label}",
            "x": round(float(cx), 2),
            "y": round(float(cy), 2),
            "type": "junction"
        })

# ==========================================
# Step 4: 对 Safe 图连通域分析 -> 生成“房间节点”(带长宽)
# ==========================================
room_labels, num_rooms = ndimage.label(safe > 0)
room_nodes = []

for i in range(1, num_rooms + 1):
    mask = (room_labels == i)
    y_c, x_c = np.where(mask)
    
    # --- 计算房间的长和宽 (边界框跨度) ---
    min_x, max_x = np.min(x_c), np.max(x_c)
    min_y, max_y = np.min(y_c), np.max(y_c)
    width_m = (max_x - min_x + 1) * res   # X轴方向的宽度
    height_m = (max_y - min_y + 1) * res  # Y轴方向的高度
    
    cx = np.mean(x_c) * res
    cy = np.mean(y_c) * res
    
    # 过滤掉面积太小（小于2平米）的碎渣区域
    pixel_count = np.sum(mask)
    area_m2 = pixel_count * (res * res)
    if area_m2 < 2.0:
        continue
        
    room_nodes.append({
        "id": f"Room_{i-1}",
        "x": round(float(cx), 2),
        "y": round(float(cy), 2),
        "width_m": round(float(width_m), 2),   # 长宽比体积直观多了
        "height_m": round(float(height_m), 2),
        "type": "room"
    })

print(f"提取到 {len(junction_nodes)} 个路口节点")
print(f"提取到 {len(room_nodes)} 个房间节点")

# ==========================================
# Step 5: 构建边 (房间连路口，路口连路口，禁止房间直接连房间)
# ==========================================
all_nodes = junction_nodes + room_nodes
node_dict = {n["id"]: n for n in all_nodes}
coords = np.array([[n["x"], n["y"]] for n in all_nodes])

edges_list = []
if len(all_nodes) > 1:
    dist_matrix = np.linalg.norm(coords[:, np.newaxis] - coords, axis=2)
    link_threshold = 3.0 
    
    for i in range(len(all_nodes)):
        for j in range(i + 1, len(all_nodes)):
            n1, n2 = all_nodes[i], all_nodes[j]
            if n1["type"] == "room" and n2["type"] == "room":
                continue
            if dist_matrix[i, j] < link_threshold:
                edges_list.append({"from": n1["id"], "to": n2["id"]})

print(f"提取到 {len(edges_list)} 条连接边")

# ==========================================
# Step 6: 可视化
# ==========================================
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(grid, cmap='gray', origin='upper', alpha=0.5)
sk_y, sk_x = np.where(skeleton > 0)
ax.scatter(sk_x, sk_y, s=0.5, color='lightgray', alpha=0.6)

# 画房间 (标注长x宽)
for n in room_nodes:
    px, py = n['x'] / res, n['y'] / res
    ax.plot(px, py, 's', markersize=12, color='dodgerblue', markeredgecolor='black', markeredgewidth=1.5)
    ax.annotate(f"{n['id']}\n{n['width_m']}x{n['height_m']}m", (px, py),
                textcoords="offset points", xytext=(15, -15), fontsize=9, color='blue', weight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

# 画路口
for n in junction_nodes:
    px, py = n['x'] / res, n['y'] / res
    ax.plot(px, py, 'o', markersize=10, color='red', markeredgecolor='black', markeredgewidth=1.5)
    ax.annotate(n['id'], (px, py), textcoords="offset points", xytext=(10, 10), fontsize=9, color='red')

# 画边
for e in edges_list:
    n1 = node_dict[e["from"]]
    n2 = node_dict[e["to"]]
    ax.plot([n1['x']/res, n2['x']/res], [n1['y']/res, n2['y']/res], 'g-', linewidth=2, alpha=0.7)

ax.set_title("Final Topology Map\n(Blue Square=Room & Size, Red Circle=Junction, Green Line=Path)", fontsize=13)
ax.set_xlim(0, W); ax.set_ylim(H, 0)
plt.tight_layout()
plt.savefig('final_topology_map.png', dpi=150, bbox_inches='tight')
plt.close()
print("可视化保存完成")

# ==========================================
# Step 7: 输出给 LLM 的 JSON
# ==========================================
topology = {"nodes": all_nodes, "edges": edges_list}

print("\n" + "=" * 50)
print("JSON for LLM:")
print("=" * 50)
print(json.dumps(topology, indent=2, ensure_ascii=False))


提取到 5 个路口节点
提取到 1 个房间节点
提取到 6 条连接边
可视化保存完成

JSON for LLM:
{
  "nodes": [
    {
      "id": "Junc_0",
      "x": 4.83,
      "y": 3.47,
      "type": "junction"
    },
    {
      "id": "Junc_1",
      "x": 6.8,
      "y": 3.7,
      "type": "junction"
    },
    {
      "id": "Junc_2",
      "x": 9.88,
      "y": 3.9,
      "type": "junction"
    },
    {
      "id": "Junc_3",
      "x": 1.73,
      "y": 4.57,
      "type": "junction"
    },
    {
      "id": "Junc_4",
      "x": 3.47,
      "y": 4.93,
      "type": "junction"
    },
    {
      "id": "Room_0",
      "x": 5.74,
      "y": 4.19,
      "width_m": 10.6,
      "height_m": 6.6,
      "type": "room"
    }
  ],
  "edges": [
    {
      "from": "Junc_0",
      "to": "Junc_1"
    },
    {
      "from": "Junc_0",
      "to": "Junc_4"
    },
    {
      "from": "Junc_0",
      "to": "Room_0"
    },
    {
      "from": "Junc_1",
      "to": "Room_0"
    },
    {
      "from": "Junc_3",
      "to": "Junc_4"
    },
    {
      "from

In [9]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.morphology import skeletonize
import json

W, H, res = 120, 80, 0.1

def make_grid():
    g = np.ones((H, W), dtype=np.uint8) * 255
    def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
    def c(cy, cx, r):
        Y, X = np.ogrid[:H, :W]
        g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0
    w(0, 3, 0, W); w(H-3, H, 0, W); w(0, H, 0, 3); w(0, H, W-3, W)
    w(3, 30, 55, 58); w(42, 77, 55, 58)
    w(3, 32, 80, 83); w(48, 77, 80, 83)
    w(74, 95, 90, 93)
    c(20, 25, 2); c(20, 35, 2); c(28, 30, 2); c(28, 40, 2)
    c(65, 12, 3)
    w(8, 22, 95, 112)
    return g

grid = make_grid()

# ==========================================
# Step 1: 基础骨架与距离图
# ==========================================
free = (grid == 255).astype(np.uint8)
dist = ndimage.distance_transform_edt(free)
skeleton = skeletonize(free).astype(np.uint8)

skel_y, skel_x = np.where(skeleton > 0)
skel_set = set(zip(skel_x.tolist(), skel_y.tolist()))

# ==========================================
# Step 2: 提取基础特征点 (端点 & 分叉点)
# ==========================================
kernel = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]])
neighbor_count = ndimage.convolve(skeleton, kernel, mode='constant', cval=0)
endpoint_mask = (neighbor_count == 1) & (skeleton > 0)
junction_mask = (neighbor_count >= 3) & (skeleton > 0)

# ==========================================
# Step 3: 核心！通过向量夹角提取拐弯点
# ==========================================
def get_neighbors(x, y, s_set):
    neis = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0: continue
            if (x+dx, y+dy) in s_set: neis.append((x+dx, y+dy))
    return neis

def walk_straight(sx, sy, prev_x, prev_y, steps, s_set):
    """沿着骨架往前走N步，遇到分叉或端点停止，返回路径"""
    path = []
    cx, cy = sx, sy
    for _ in range(steps):
        neis = get_neighbors(cx, cy, s_set)
        neis = [n for n in neis if n != (prev_x, prev_y)]
        if len(neis) != 1: break 
        path.append(neis[0])
        prev_x, prev_y = cx, cy
        cx, cy = neis[0]
    return path

bend_points_set = set()
ANGLE_THRESHOLD = 30 # 方向改变超过30度视为拐弯

for x, y in skel_set:
    neis = get_neighbors(x, y, skel_set)
    if len(neis) != 2: continue # 只处理度数为2的普通线段点
    
    n1, n2 = neis
    # 向两边各探索4步，拉长向量以过滤像素锯齿
    path1 = walk_straight(n1[0], n1[1], x, y, 4, skel_set)
    path2 = walk_straight(n2[0], n2[1], x, y, 4, skel_set)
    
    if len(path1) > 0 and len(path2) > 0:
        v1 = np.array(path1[-1]) - np.array([x, y])
        v2 = np.array(path2[-1]) - np.array([x, y])
        n1_norm, n2_norm = np.linalg.norm(v1), np.linalg.norm(v2)
        
        if n1_norm > 0 and n2_norm > 0:
            cos_angle = np.dot(v1, v2) / (n1_norm * n2_norm)
            angle = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
            if angle < (180 - ANGLE_THRESHOLD):
                bend_points_set.add((x, y))

# ==========================================
# Step 4: 合并所有关键节点并计算 Voronoi Span
# ==========================================
key_points_set = set()
ep_y, ep_x = np.where(endpoint_mask)
for x, y in zip(ep_x, ep_y): key_points_set.add((x, y))
jp_y, jp_x = np.where(junction_mask)
for x, y in zip(jp_x, jp_y): key_points_set.add((x, y))
key_points_set.update(bend_points_set)

def calc_local_span(x, y, s_set, radius=12):
    """提取节点周围局部骨架的X/Y跨度，代表该处的空间范围"""
    local_pts = []
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            if (x+dx, y+dy) in s_set:
                local_pts.append((x+dx, y+dy))
    if not local_pts: return 0.0, 0.0
    pts = np.array(local_pts)
    # X和Y的极差 * 分辨率 = 物理跨度
    span_x = (np.max(pts[:, 0]) - np.min(pts[:, 0])) * res
    span_y = (np.max(pts[:, 1]) - np.min(pts[:, 1])) * res
    return round(float(span_x), 2), round(float(span_y), 2)

nodes = []
node_id_map = {}
idx = 0
for x, y in key_points_set:
    sx, sy = calc_local_span(x, y, skel_set)
    node_id = f"N_{idx}"
    nodes.append({
        "id": node_id,
        "x": round(float(x * res), 2),
        "y": round(float(y * res), 2),
        "span_x": sx, # 局部X方向物理跨度(米)
        "span_y": sy  # 局部Y方向物理跨度(米)
    })
    node_id_map[(x, y)] = node_id
    idx += 1

print(f"提取到 {len(nodes)} 个纯导航节点 (含拐点)")

# ==========================================
# Step 5: 构建边 (切断节点后的连通域连线)
# ==========================================
cut_skeleton = skeleton.copy()
for x, y in key_points_set:
    cut_skeleton[y, x] = 0 # 把节点从骨架上挖空，剩下纯粹的直线段

labels, num = ndimage.label(cut_skeleton > 0)
edges_list = set()

if num > 0:
    # 建立剩余骨架像素到连通域ID的映射
    skel_label_map = {}
    ly, lx = np.where(labels > 0)
    for y, x in zip(ly, lx):
        skel_label_map[(x, y)] = labels[y, x]
        
    # 寻找每个节点连接了哪些直线段(连通域)
    label_to_nodes = {}
    for x, y in key_points_set:
        for nx, ny in get_neighbors(x, y, skel_set):
            if (nx, ny) in skel_label_map:
                l = skel_label_map[(nx, ny)]
                if l not in label_to_nodes: label_to_nodes[l] = []
                label_to_nodes[l].append(node_id_map[(x, y)])
                
    # 同一直线段两端的关键点互相连线
    for l, connected_node_ids in label_to_nodes.items():
        connected_node_ids = list(set(connected_node_ids))
        if len(connected_node_ids) == 2:
            edge = tuple(sorted(connected_node_ids))
            edges_list.add(edge)
        # 理论上切断后，一段直线只能连接2个节点

edges_json = [{"from": e[0], "to": e[1]} for e in edges_list]
print(f"提取到 {len(edges_json)} 条直线路径边")

# ==========================================
# Step 6: 可视化
# ==========================================
node_dict = {n["id"]: n for n in nodes}

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(grid, cmap='gray', origin='upper', alpha=0.5)
ax.scatter(skel_x, skel_y, s=0.5, color='lightgray', alpha=0.6)

for n in nodes:
    px, py = n['x'] / res, n['y'] / res
    # 画节点，大小随 span 面积缩放，直观反映空间大小
    size = (n['span_x'] * n['span_y']) * 8 
    ax.plot(px, py, 'o', markersize=max(4, min(size, 18)), color='red', markeredgecolor='black')
    # 标注 span
    ax.annotate(f"{n['span_x']}x{n['span_y']}", (px, py),
                textcoords="offset points", xytext=(8, 8), fontsize=7, color='darkred')

for e in edges_json:
    n1, n2 = node_dict[e["from"]], node_dict[e["to"]]
    ax.plot([n1['x']/res, n2['x']/res], [n1['y']/res, n2['y']/res], 'g-', linewidth=2, alpha=0.8)

ax.set_title("Pure Voronoi Span Topology\n(Red Dot=Node, Size/Text=Local Span, Green=Straight Path)", fontsize=13)
ax.set_xlim(0, W); ax.set_ylim(H, 0)
plt.tight_layout()
plt.savefig('pure_span_topology.png', dpi=150, bbox_inches='tight')
plt.close()

# ==========================================
# Step 7: 输出 LLM JSON
# ==========================================
print("\n" + "=" * 50)
print("JSON for LLM:")
print("=" * 50)
print(json.dumps({"nodes": nodes, "edges": edges_json}, indent=2, ensure_ascii=False))


提取到 116 个纯导航节点 (含拐点)
提取到 31 条直线路径边

JSON for LLM:
{
  "nodes": [
    {
      "id": "N_0",
      "x": 4.8,
      "y": 2.1,
      "span_x": 1.2,
      "span_y": 2.3
    },
    {
      "id": "N_1",
      "x": 9.9,
      "y": 3.9,
      "span_x": 2.3,
      "span_y": 2.4
    },
    {
      "id": "N_2",
      "x": 0.7,
      "y": 7.2,
      "span_x": 1.3,
      "span_y": 1.2
    },
    {
      "id": "N_3",
      "x": 1.8,
      "y": 7.2,
      "span_x": 2.4,
      "span_y": 1.2
    },
    {
      "id": "N_4",
      "x": 11.4,
      "y": 2.1,
      "span_x": 1.0,
      "span_y": 2.4
    },
    {
      "id": "N_5",
      "x": 1.3,
      "y": 3.3,
      "span_x": 1.2,
      "span_y": 2.4
    },
    {
      "id": "N_6",
      "x": 8.6,
      "y": 6.9,
      "span_x": 1.3,
      "span_y": 1.6
    },
    {
      "id": "N_7",
      "x": 3.0,
      "y": 4.6,
      "span_x": 2.4,
      "span_y": 2.4
    },
    {
      "id": "N_8",
      "x": 1.5,
      "y": 3.6,
      "span_x": 1.4,
      "span_y": 

In [35]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.morphology import skeletonize
import json

W, H, res = 120, 80, 0.1

def make_grid():
    g = np.ones((H, W), dtype=np.uint8) * 255
    def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
    def c(cy, cx, r):
        Y, X = np.ogrid[:H, :W]
        g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0
    w(0, 3, 0, W); w(H-3, H, 0, W); w(0, H, 0, 3); w(0, H, W-3, W)
    w(3, 30, 55, 58); w(42, 77, 55, 58)
    w(3, 32, 80, 83); w(48, 77, 80, 83)
    w(74, 95, 90, 93)
    c(20, 25, 2); c(20, 35, 2); c(28, 30, 2); c(28, 40, 2)
    c(65, 12, 3)
    w(8, 22, 95, 112)
    return g

grid = make_grid()

# ==========================================
# Step 1: 基础骨架与距离图
# ==========================================
free = (grid == 255).astype(np.uint8)
dist = ndimage.distance_transform_edt(free)
skeleton = skeletonize(free).astype(np.uint8)

skel_y, skel_x = np.where(skeleton > 0)
skel_set = set(zip(skel_x.tolist(), skel_y.tolist()))

# ==========================================
# Step 2: 提取基础特征点 (端点 & 分叉点)
# ==========================================
kernel = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]])
neighbor_count = ndimage.convolve(skeleton, kernel, mode='constant', cval=0)
endpoint_mask = (neighbor_count == 1) & (skeleton > 0)
junction_mask = (neighbor_count >= 3) & (skeleton > 0)

# ==========================================
# Step 3: 优化！通过向量夹角提取拐弯点
# ==========================================
def get_neighbors(x, y, s_set):
    neis = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0: continue
            if (x+dx, y+dy) in s_set: neis.append((x+dx, y+dy))
    return neis

def walk_straight(sx, sy, prev_x, prev_y, steps, s_set):
    """沿着骨架往前走N步，遇到分叉或端点停止，返回路径"""
    path = []
    cx, cy = sx, sy
    for _ in range(steps):
        neis = get_neighbors(cx, cy, s_set)
        neis = [n for n in neis if n != (prev_x, prev_y)]
        if len(neis) != 1: break 
        path.append(neis[0])
        prev_x, prev_y = cx, cy
        cx, cy = neis[0]
    return path

bend_points_set = set()
ANGLE_THRESHOLD = 60 # 增加到60度，大幅减少拐点
MAX_STRAIGHT_STEPS = 12 # 增加到12步，确保能穿透长走廊

for x, y in skel_set:
    neis = get_neighbors(x, y, skel_set)
    if len(neis) != 2: continue # 只处理度数为2的普通线段点
    
    n1, n2 = neis
    # 向两边各探索更多步，确保能穿透长走廊
    path1 = walk_straight(n1[0], n1[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    path2 = walk_straight(n2[0], n2[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    
    if len(path1) > 0 and len(path2) > 0:
        v1 = np.array(path1[-1]) - np.array([x, y])
        v2 = np.array(path2[-1]) - np.array([x, y])
        n1_norm, n2_norm = np.linalg.norm(v1), np.linalg.norm(v2)
        
        if n1_norm > 0 and n2_norm > 0:
            cos_angle = np.dot(v1, v2) / (n1_norm * n2_norm)
            angle = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
            if angle < (180 - ANGLE_THRESHOLD):
                bend_points_set.add((x, y))

# ==========================================
# Step 4: 合并所有关键节点并计算 Voronoi Span
# ==========================================
key_points_set = set()
ep_y, ep_x = np.where(endpoint_mask)
for x, y in zip(ep_x, ep_y): key_points_set.add((x, y))
jp_y, jp_x = np.where(junction_mask)
for x, y in zip(jp_x, jp_y): key_points_set.add((x, y))
key_points_set.update(bend_points_set)

def calc_local_span(x, y, s_set, radius=25):
    """提取节点周围局部骨架的X/Y跨度，代表该处的空间范围"""
    local_pts = []
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            if (x+dx, y+dy) in s_set:
                local_pts.append((x+dx, y+dy))
    if not local_pts: return 0.0, 0.0
    pts = np.array(local_pts)
    # X和Y的极差 * 分辨率 = 物理跨度
    span_x = (np.max(pts[:, 0]) - np.min(pts[:, 0])) * res
    span_y = (np.max(pts[:, 1]) - np.min(pts[:, 1])) * res
    return round(float(span_x), 2), round(float(span_y), 2)

nodes = []
node_id_map = {}
idx = 0
for x, y in key_points_set:
    sx, sy = calc_local_span(x, y, skel_set)
    node_id = f"N_{idx}"
    nodes.append({
        "id": node_id,
        "x": round(float(x * res), 2),
        "y": round(float(y * res), 2),
        "span_x": sx, # 局部X方向物理跨度(米)
        "span_y": sy  # 局部Y方向物理跨度(米)
    })
    node_id_map[(x, y)] = node_id
    idx += 1

print(f"提取到 {len(nodes)} 个纯导航节点 (含拐点)")

# ==========================================
# Step 5: 构建边 (切断节点后的连通域连线)
# ==========================================
cut_skeleton = skeleton.copy()
for x, y in key_points_set:
    cut_skeleton[y, x] = 0 # 把节点从骨架上挖空，剩下纯粹的直线段

labels, num = ndimage.label(cut_skeleton > 0)
edges_list = set()

if num > 0:
    # 建立剩余骨架像素到连通域ID的映射
    skel_label_map = {}
    ly, lx = np.where(labels > 0)
    for y, x in zip(ly, lx):
        skel_label_map[(x, y)] = labels[y, x]
        
    # 寻找每个节点连接了哪些直线段(连通域)
    label_to_nodes = {}
    for x, y in key_points_set:
        for nx, ny in get_neighbors(x, y, skel_set):
            if (nx, ny) in skel_label_map:
                l = skel_label_map[(nx, ny)]
                if l not in label_to_nodes: label_to_nodes[l] = []
                label_to_nodes[l].append(node_id_map[(x, y)])
                
    # 同一直线段两端的关键点互相连线
    for l, connected_node_ids in label_to_nodes.items():
        connected_node_ids = list(set(connected_node_ids))
        if len(connected_node_ids) == 2:
            edge = tuple(sorted(connected_node_ids))
            edges_list.add(edge)
        # 理论上切断后，一段直线只能连接2个节点

edges_json = [{"from": e[0], "to": e[1]} for e in edges_list]
print(f"提取到 {len(edges_json)} 条直线路径边")

# ==========================================
# Step 6: 可视化
# ==========================================
node_dict = {n["id"]: n for n in nodes}

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(grid, cmap='gray', origin='upper', alpha=0.5)
ax.scatter(skel_x, skel_y, s=0.5, color='lightgray', alpha=0.6)

for n in nodes:
    px, py = n['x'] / res, n['y'] / res
    # 画节点，大小随 span 面积缩放，直观反映空间大小
    size = (n['span_x'] * n['span_y']) * 8 
    ax.plot(px, py, 'o', markersize=max(4, min(size, 18)), color='red', markeredgecolor='black')
    # 标注 span
    ax.annotate(f"{n['span_x']}x{n['span_y']}", (px, py),
                textcoords="offset points", xytext=(8, 8), fontsize=7, color='darkred')

for e in edges_json:
    n1, n2 = node_dict[e["from"]], node_dict[e["to"]]
    ax.plot([n1['x']/res, n2['x']/res], [n1['y']/res, n2['y']/res], 'g-', linewidth=2, alpha=0.8)

ax.set_title("Pure Voronoi Span Topology\n(Red Dot=Node, Size/Text=Local Span, Green=Straight Path)", fontsize=13)
ax.set_xlim(0, W); ax.set_ylim(H, 0)
plt.tight_layout()
plt.savefig('pure_span_topology_final.png', dpi=150, bbox_inches='tight')
plt.close()

# ==========================================
# Step 7: 输出 LLM JSON
# ==========================================
print("\n" + "=" * 50)
print("JSON for LLM:")
print("=" * 50)
print(json.dumps({"nodes": nodes, "edges": edges_json}, indent=2, ensure_ascii=False))


提取到 100 个纯导航节点 (含拐点)
提取到 14 条直线路径边

JSON for LLM:
{
  "nodes": [
    {
      "id": "N_0",
      "x": 11.0,
      "y": 0.5,
      "span_x": 2.6,
      "span_y": 2.5
    },
    {
      "id": "N_1",
      "x": 9.9,
      "y": 3.9,
      "span_x": 4.0,
      "span_y": 5.0
    },
    {
      "id": "N_2",
      "x": 0.7,
      "y": 7.2,
      "span_x": 2.6,
      "span_y": 2.5
    },
    {
      "id": "N_3",
      "x": 1.5,
      "y": 3.6,
      "span_x": 3.4,
      "span_y": 5.0
    },
    {
      "id": "N_4",
      "x": 4.8,
      "y": 3.6,
      "span_x": 5.0,
      "span_y": 5.0
    },
    {
      "id": "N_5",
      "x": 3.5,
      "y": 4.8,
      "span_x": 5.0,
      "span_y": 4.9
    },
    {
      "id": "N_6",
      "x": 3.6,
      "y": 2.5,
      "span_x": 4.8,
      "span_y": 4.0
    },
    {
      "id": "N_7",
      "x": 8.5,
      "y": 7.3,
      "span_x": 3.7,
      "span_y": 2.5
    },
    {
      "id": "N_8",
      "x": 0.9,
      "y": 7.2,
      "span_x": 2.8,
      "span_y": 

In [36]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.morphology import skeletonize
import json

W, H, res = 200, 150, 0.1  # 增大地图尺寸以容纳更多复杂结构

def make_complex_grid():
    g = np.ones((H, W), dtype=np.uint8) * 255
    def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
    def c(cy, cx, r):
        Y, X = np.ogrid[:H, :W]
        g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0
    
    # 外墙
    w(0, 3, 0, W); w(H-3, H, 0, W); w(0, H, 0, 3); w(0, H, W-3, W)
    
    # 主走廊
    w(30, 120, 40, 60)
    w(30, 120, 140, 160)
    
    # 左侧房间
    w(10, 30, 10, 40)
    w(10, 30, 70, 100)
    w(10, 30, 120, 150)
    
    # 右侧房间
    w(10, 30, 170, 200)
    w(10, 30, 220, 250)
    
    # 中间房间
    w(50, 70, 80, 120)
    w(50, 70, 180, 220)
    
    # 上方房间
    w(80, 100, 50, 90)
    w(80, 100, 130, 170)
    w(80, 100, 210, 250)
    
    # 下方房间
    w(120, 140, 70, 110)
    w(120, 140, 150, 190)
    w(120, 140, 230, 270)
    
    # 添加一些圆形障碍物
    c(25, 25, 3)
    c(25, 175, 3)
    c(125, 25, 3)
    c(125, 175, 3)
    c(75, 75, 4)
    c(175, 75, 4)
    c(75, 125, 4)
    c(175, 125, 4)
    
    # 添加一些不规则形状
    w(60, 80, 30, 50)
    w(60, 80, 190, 210)
    w(100, 120, 30, 50)
    w(100, 120, 190, 210)
    
    return g

grid = make_complex_grid()

# ==========================================
# Step 1: 基础骨架与距离图
# ==========================================
free = (grid == 255).astype(np.uint8)
dist = ndimage.distance_transform_edt(free)
skeleton = skeletonize(free).astype(np.uint8)

skel_y, skel_x = np.where(skeleton > 0)
skel_set = set(zip(skel_x.tolist(), skel_y.tolist()))

# ==========================================
# Step 2: 提取基础特征点 (端点 & 分叉点)
# ==========================================
kernel = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]])
neighbor_count = ndimage.convolve(skeleton, kernel, mode='constant', cval=0)
endpoint_mask = (neighbor_count == 1) & (skeleton > 0)
junction_mask = (neighbor_count >= 3) & (skeleton > 0)

# ==========================================
# Step 3: 通过向量夹角提取拐弯点
# ==========================================
def get_neighbors(x, y, s_set):
    neis = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0: continue
            if (x+dx, y+dy) in s_set: neis.append((x+dx, y+dy))
    return neis

def walk_straight(sx, sy, prev_x, prev_y, steps, s_set):
    """沿着骨架往前走N步，遇到分叉或端点停止，返回路径"""
    path = []
    cx, cy = sx, sy
    for _ in range(steps):
        neis = get_neighbors(cx, cy, s_set)
        neis = [n for n in neis if n != (prev_x, prev_y)]
        if len(neis) != 1: break 
        path.append(neis[0])
        prev_x, prev_y = cx, cy
        cx, cy = neis[0]
    return path

bend_points_set = set()
ANGLE_THRESHOLD = 45 # 60度阈值
MAX_STRAIGHT_STEPS = 15 # 15步确保能穿透长走廊

for x, y in skel_set:
    neis = get_neighbors(x, y, skel_set)
    if len(neis) != 2: continue # 只处理度数为2的普通线段点
    
    n1, n2 = neis
    # 向两边各探索更多步，确保能穿透长走廊
    path1 = walk_straight(n1[0], n1[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    path2 = walk_straight(n2[0], n2[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    
    if len(path1) > 0 and len(path2) > 0:
        v1 = np.array(path1[-1]) - np.array([x, y])
        v2 = np.array(path2[-1]) - np.array([x, y])
        n1_norm, n2_norm = np.linalg.norm(v1), np.linalg.norm(v2)
        
        if n1_norm > 0 and n2_norm > 0:
            cos_angle = np.dot(v1, v2) / (n1_norm * n2_norm)
            angle = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
            if angle < (180 - ANGLE_THRESHOLD):
                bend_points_set.add((x, y))

# ==========================================
# Step 4: 合并所有关键节点并计算 Voronoi Span
# ==========================================
key_points_set = set()
ep_y, ep_x = np.where(endpoint_mask)
for x, y in zip(ep_x, ep_y): key_points_set.add((x, y))
jp_y, jp_x = np.where(junction_mask)
for x, y in zip(jp_x, jp_y): key_points_set.add((x, y))
key_points_set.update(bend_points_set)

def calc_local_span(x, y, s_set, radius=30):
    """提取节点周围局部骨架的X/Y跨度，代表该处的空间范围"""
    local_pts = []
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            if (x+dx, y+dy) in s_set:
                local_pts.append((x+dx, y+dy))
    if not local_pts: return 0.0, 0.0
    pts = np.array(local_pts)
    # X和Y的极差 * 分辨率 = 物理跨度
    span_x = (np.max(pts[:, 0]) - np.min(pts[:, 0])) * res
    span_y = (np.max(pts[:, 1]) - np.min(pts[:, 1])) * res
    return round(float(span_x), 2), round(float(span_y), 2)

nodes = []
node_id_map = {}
idx = 0
for x, y in key_points_set:
    sx, sy = calc_local_span(x, y, skel_set)
    node_id = f"N_{idx}"
    nodes.append({
        "id": node_id,
        "x": round(float(x * res), 2),
        "y": round(float(y * res), 2),
        "span_x": sx, # 局部X方向物理跨度(米)
        "span_y": sy  # 局部Y方向物理跨度(米)
    })
    node_id_map[(x, y)] = node_id
    idx += 1

print(f"提取到 {len(nodes)} 个纯导航节点 (含拐点)")

# ==========================================
# Step 5: 构建边 (切断节点后的连通域连线)
# ==========================================
cut_skeleton = skeleton.copy()
for x, y in key_points_set:
    cut_skeleton[y, x] = 0 # 把节点从骨架上挖空，剩下纯粹的直线段

labels, num = ndimage.label(cut_skeleton > 0)
edges_list = set()

if num > 0:
    # 建立剩余骨架像素到连通域ID的映射
    skel_label_map = {}
    ly, lx = np.where(labels > 0)
    for y, x in zip(ly, lx):
        skel_label_map[(x, y)] = labels[y, x]
        
    # 寻找每个节点连接了哪些直线段(连通域)
    label_to_nodes = {}
    for x, y in key_points_set:
        for nx, ny in get_neighbors(x, y, skel_set):
            if (nx, ny) in skel_label_map:
                l = skel_label_map[(nx, ny)]
                if l not in label_to_nodes: label_to_nodes[l] = []
                label_to_nodes[l].append(node_id_map[(x, y)])
                
    # 同一直线段两端的关键点互相连线
    for l, connected_node_ids in label_to_nodes.items():
        connected_node_ids = list(set(connected_node_ids))
        if len(connected_node_ids) == 2:
            edge = tuple(sorted(connected_node_ids))
            edges_list.add(edge)
        # 理论上切断后，一段直线只能连接2个节点

edges_json = [{"from": e[0], "to": e[1]} for e in edges_list]
print(f"提取到 {len(edges_json)} 条直线路径边")

# ==========================================
# Step 6: 可视化
# ==========================================
node_dict = {n["id"]: n for n in nodes}

fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(grid, cmap='gray', origin='upper', alpha=0.5)
ax.scatter(skel_x, skel_y, s=0.5, color='lightgray', alpha=0.6)

for n in nodes:
    px, py = n['x'] / res, n['y'] / res
    # 画节点，大小随 span 面积缩放，直观反映空间大小
    size = (n['span_x'] * n['span_y']) * 8 
    ax.plot(px, py, 'o', markersize=max(4, min(size, 18)), color='red', markeredgecolor='black')
    # 标注 span
    ax.annotate(f"{n['span_x']}x{n['span_y']}", (px, py),
                textcoords="offset points", xytext=(8, 8), fontsize=7, color='darkred')

for e in edges_json:
    n1, n2 = node_dict[e["from"]], node_dict[e["to"]]
    ax.plot([n1['x']/res, n2['x']/res], [n1['y']/res, n2['y']/res], 'g-', linewidth=2, alpha=0.8)

ax.set_title("Complex Map Voronoi Span Topology\n(Red Dot=Node, Size/Text=Local Span, Green=Straight Path)", fontsize=13)
ax.set_xlim(0, W); ax.set_ylim(H, 0)
plt.tight_layout()
plt.savefig('complex_map_topology.png', dpi=150, bbox_inches='tight')
plt.close()

# ==========================================
# Step 7: 输出 LLM JSON
# ==========================================
print("\n" + "=" * 50)
print("JSON for LLM:")
print("=" * 50)
print(json.dumps({"nodes": nodes, "edges": edges_json}, indent=2, ensure_ascii=False))


提取到 184 个纯导航节点 (含拐点)
提取到 17 条直线路径边

JSON for LLM:
{
  "nodes": [
    {
      "id": "N_0",
      "x": 12.9,
      "y": 4.3,
      "span_x": 6.0,
      "span_y": 6.0
    },
    {
      "id": "N_1",
      "x": 12.1,
      "y": 3.9,
      "span_x": 4.3,
      "span_y": 6.0
    },
    {
      "id": "N_2",
      "x": 18.7,
      "y": 3.9,
      "span_x": 3.0,
      "span_y": 6.0
    },
    {
      "id": "N_3",
      "x": 17.9,
      "y": 9.0,
      "span_x": 2.2,
      "span_y": 6.0
    },
    {
      "id": "N_4",
      "x": 1.2,
      "y": 13.2,
      "span_x": 3.0,
      "span_y": 3.5
    },
    {
      "id": "N_5",
      "x": 13.4,
      "y": 6.7,
      "span_x": 3.0,
      "span_y": 6.0
    },
    {
      "id": "N_6",
      "x": 17.8,
      "y": 10.9,
      "span_x": 2.5,
      "span_y": 6.0
    },
    {
      "id": "N_7",
      "x": 17.9,
      "y": 10.8,
      "span_x": 2.5,
      "span_y": 6.0
    },
    {
      "id": "N_8",
      "x": 1.0,
      "y": 0.6,
      "span_x": 3.4,
      "

In [38]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.morphology import skeletonize
import json

W, H, res = 200, 150, 0.1  # 增大地图尺寸以容纳更多复杂结构

def make_complex_grid():
    g = np.ones((H, W), dtype=np.uint8) * 255
    def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
    def c(cy, cx, r):
        Y, X = np.ogrid[:H, :W]
        g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0
    
    # 外墙
    w(0, 3, 0, W); w(H-3, H, 0, W); w(0, H, 0, 3); w(0, H, W-3, W)
    
    # 主走廊
    w(30, 120, 40, 60)
    w(30, 120, 140, 160)
    
    # 左侧房间
    w(10, 30, 10, 40)
    w(10, 30, 70, 100)
    w(10, 30, 120, 150)
    
    # 右侧房间
    w(10, 30, 170, 200)
    w(10, 30, 220, 250)
    
    # 中间房间
    w(50, 70, 80, 120)
    w(50, 70, 180, 220)
    
    # 上方房间
    w(80, 100, 50, 90)
    w(80, 100, 130, 170)
    w(80, 100, 210, 250)
    
    # 下方房间
    w(120, 140, 70, 110)
    w(120, 140, 150, 190)
    w(120, 140, 230, 270)
    
    # 添加一些圆形障碍物
    c(25, 25, 3)
    c(25, 175, 3)
    c(125, 25, 3)
    c(125, 175, 3)
    c(75, 75, 4)
    c(175, 75, 4)
    c(75, 125, 4)
    c(175, 125, 4)
    
    # 添加一些不规则形状
    w(60, 80, 30, 50)
    w(60, 80, 190, 210)
    w(100, 120, 30, 50)
    w(100, 120, 190, 210)
    
    return g

grid = make_complex_grid()

# ==========================================
# Step 1: 基础骨架与距离图
# ==========================================
free = (grid == 255).astype(np.uint8)
dist = ndimage.distance_transform_edt(free)
skeleton = skeletonize(free).astype(np.uint8)

skel_y, skel_x = np.where(skeleton > 0)
skel_set = set(zip(skel_x.tolist(), skel_y.tolist()))

# ==========================================
# Step 2: 提取基础特征点 (端点 & 分叉点)
# ==========================================
kernel = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]])
neighbor_count = ndimage.convolve(skeleton, kernel, mode='constant', cval=0)
endpoint_mask = (neighbor_count == 1) & (skeleton > 0)
junction_mask = (neighbor_count >= 3) & (skeleton > 0)

# ==========================================
# Step 3: 通过向量夹角提取拐弯点
# ==========================================
def get_neighbors(x, y, s_set):
    neis = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0: continue
            if (x+dx, y+dy) in s_set: neis.append((x+dx, y+dy))
    return neis

def walk_straight(sx, sy, prev_x, prev_y, steps, s_set):
    """沿着骨架往前走N步，遇到分叉或端点停止，返回路径"""
    path = []
    cx, cy = sx, sy
    for _ in range(steps):
        neis = get_neighbors(cx, cy, s_set)
        neis = [n for n in neis if n != (prev_x, prev_y)]
        if len(neis) != 1: break 
        path.append(neis[0])
        prev_x, prev_y = cx, cy
        cx, cy = neis[0]
    return path

bend_points_set = set()
ANGLE_THRESHOLD = 60 # 60度阈值
MAX_STRAIGHT_STEPS = 15 # 15步确保能穿透长走廊

for x, y in skel_set:
    neis = get_neighbors(x, y, skel_set)
    if len(neis) != 2: continue # 只处理度数为2的普通线段点
    
    n1, n2 = neis
    # 向两边各探索更多步，确保能穿透长走廊
    path1 = walk_straight(n1[0], n1[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    path2 = walk_straight(n2[0], n2[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    
    if len(path1) > 0 and len(path2) > 0:
        v1 = np.array(path1[-1]) - np.array([x, y])
        v2 = np.array(path2[-1]) - np.array([x, y])
        n1_norm, n2_norm = np.linalg.norm(v1), np.linalg.norm(v2)
        
        if n1_norm > 0 and n2_norm > 0:
            cos_angle = np.dot(v1, v2) / (n1_norm * n2_norm)
            angle = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
            if angle < (180 - ANGLE_THRESHOLD):
                bend_points_set.add((x, y))

# ==========================================
# Step 4: 合并所有关键节点并计算 Voronoi Span
# ==========================================
key_points_set = set()
ep_y, ep_x = np.where(endpoint_mask)
for x, y in zip(ep_x, ep_y): key_points_set.add((x, y))
jp_y, jp_x = np.where(junction_mask)
for x, y in zip(jp_x, jp_y): key_points_set.add((x, y))
key_points_set.update(bend_points_set)

def calc_local_span(x, y, s_set, radius=30):
    """提取节点周围局部骨架的X/Y跨度，代表该处的空间范围"""
    local_pts = []
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            if (x+dx, y+dy) in s_set:
                local_pts.append((x+dx, y+dy))
    if not local_pts: return 0.0, 0.0
    pts = np.array(local_pts)
    # X和Y的极差 * 分辨率 = 物理跨度
    span_x = (np.max(pts[:, 0]) - np.min(pts[:, 0])) * res
    span_y = (np.max(pts[:, 1]) - np.min(pts[:, 1])) * res
    return round(float(span_x), 2), round(float(span_y), 2)

nodes = []
node_id_map = {}
idx = 0
for x, y in key_points_set:
    sx, sy = calc_local_span(x, y, skel_set)
    node_id = f"N_{idx}"
    nodes.append({
        "id": node_id,
        "x": round(float(x * res), 2),
        "y": round(float(y * res), 2),
        "span_x": sx, # 局部X方向物理跨度(米)
        "span_y": sy  # 局部Y方向物理跨度(米)
    })
    node_id_map[(x, y)] = node_id
    idx += 1

print(f"提取到 {len(nodes)} 个纯导航节点 (含拐点)")

# ==========================================
# Step 5: 构建边 (切断节点后的连通域连线)
# ==========================================
cut_skeleton = skeleton.copy()
for x, y in key_points_set:
    cut_skeleton[y, x] = 0 # 把节点从骨架上挖空，剩下纯粹的直线段

labels, num = ndimage.label(cut_skeleton > 0)
edges_list = set()

if num > 0:
    # 建立剩余骨架像素到连通域ID的映射
    skel_label_map = {}
    ly, lx = np.where(labels > 0)
    for y, x in zip(ly, lx):
        skel_label_map[(x, y)] = labels[y, x]
        
    # 寻找每个节点连接了哪些直线段(连通域)
    label_to_nodes = {}
    for x, y in key_points_set:
        for nx, ny in get_neighbors(x, y, skel_set):
            if (nx, ny) in skel_label_map:
                l = skel_label_map[(nx, ny)]
                if l not in label_to_nodes: label_to_nodes[l] = []
                label_to_nodes[l].append(node_id_map[(x, y)])
                
    # 同一直线段两端的关键点互相连线
    for l, connected_node_ids in label_to_nodes.items():
        connected_node_ids = list(set(connected_node_ids))
        if len(connected_node_ids) == 2:
            edge = tuple(sorted(connected_node_ids))
            edges_list.add(edge)
        # 理论上切断后，一段直线只能连接2个节点

edges_json = [{"from": e[0], "to": e[1]} for e in edges_list]
print(f"提取到 {len(edges_json)} 条直线路径边")

# ==========================================
# Step 6: 可视化
# ==========================================
node_dict = {n["id"]: n for n in nodes}

fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(grid, cmap='gray', origin='upper', alpha=0.5)
ax.scatter(skel_x, skel_y, s=0.5, color='lightgray', alpha=0.6)

for n in nodes:
    px, py = n['x'] / res, n['y'] / res
    # 画节点，大小随 span 面积缩放，直观反映空间大小
    size = (n['span_x'] * n['span_y']) * 8 
    ax.plot(px, py, 'o', markersize=max(4, min(size, 18)), color='red', markeredgecolor='black')
    # 标注 span
    ax.annotate(f"{n['span_x']}x{n['span_y']}", (px, py),
                textcoords="offset points", xytext=(8, 8), fontsize=7, color='darkred')

for e in edges_json:
    n1, n2 = node_dict[e["from"]], node_dict[e["to"]]
    ax.plot([n1['x']/res, n2['x']/res], [n1['y']/res, n2['y']/res], 'g-', linewidth=2, alpha=0.8)

ax.set_title("Complex Map Voronoi Span Topology\n(Red Dot=Node, Size/Text=Local Span, Green=Straight Path)", fontsize=13)
ax.set_xlim(0, W); ax.set_ylim(H, 0)
plt.tight_layout()
plt.savefig('complex_map_topology.png', dpi=150, bbox_inches='tight')
plt.close()

# ==========================================
# Step 7: 输出 LLM JSON
# ==========================================
print("\n" + "=" * 50)
print("JSON for LLM:")
print("=" * 50)
print(json.dumps({"nodes": nodes, "edges": edges_json}, indent=2, ensure_ascii=False))


提取到 149 个纯导航节点 (含拐点)
提取到 18 条直线路径边

JSON for LLM:
{
  "nodes": [
    {
      "id": "N_0",
      "x": 12.9,
      "y": 4.3,
      "span_x": 6.0,
      "span_y": 6.0
    },
    {
      "id": "N_1",
      "x": 18.7,
      "y": 3.9,
      "span_x": 3.0,
      "span_y": 6.0
    },
    {
      "id": "N_2",
      "x": 17.9,
      "y": 9.0,
      "span_x": 2.2,
      "span_y": 6.0
    },
    {
      "id": "N_3",
      "x": 1.2,
      "y": 13.2,
      "span_x": 3.0,
      "span_y": 3.5
    },
    {
      "id": "N_4",
      "x": 13.4,
      "y": 6.7,
      "span_x": 3.0,
      "span_y": 6.0
    },
    {
      "id": "N_5",
      "x": 17.8,
      "y": 10.9,
      "span_x": 2.5,
      "span_y": 6.0
    },
    {
      "id": "N_6",
      "x": 17.9,
      "y": 10.8,
      "span_x": 2.5,
      "span_y": 6.0
    },
    {
      "id": "N_7",
      "x": 1.0,
      "y": 0.6,
      "span_x": 3.4,
      "span_y": 3.0
    },
    {
      "id": "N_8",
      "x": 16.0,
      "y": 1.2,
      "span_x": 6.0,
      "

In [39]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.morphology import skeletonize
import json
from sklearn.cluster import DBSCAN

W, H, res = 200, 150, 0.1

def make_complex_grid():
    g = np.ones((H, W), dtype=np.uint8) * 255
    def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
    def c(cy, cx, r):
        Y, X = np.ogrid[:H, :W]
        g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0
    
    # 外墙
    w(0, 3, 0, W); w(H-3, H, 0, W); w(0, H, 0, 3); w(0, H, W-3, W)
    
    # 主走廊
    w(30, 120, 40, 60)
    w(30, 120, 140, 160)
    
    # 左侧房间
    w(10, 30, 10, 40)
    w(10, 30, 70, 100)
    w(10, 30, 120, 150)
    
    # 右侧房间
    w(10, 30, 170, 200)
    w(10, 30, 220, 250)
    
    # 中间房间
    w(50, 70, 80, 120)
    w(50, 70, 180, 220)
    
    # 上方房间
    w(80, 100, 50, 90)
    w(80, 100, 130, 170)
    w(80, 100, 210, 250)
    
    # 下方房间
    w(120, 140, 70, 110)
    w(120, 140, 150, 190)
    w(120, 140, 230, 270)
    
    # 添加一些圆形障碍物
    c(25, 25, 3)
    c(25, 175, 3)
    c(125, 25, 3)
    c(125, 175, 3)
    c(75, 75, 4)
    c(175, 75, 4)
    c(75, 125, 4)
    c(175, 125, 4)
    
    # 添加一些不规则形状
    w(60, 80, 30, 50)
    w(60, 80, 190, 210)
    w(100, 120, 30, 50)
    w(100, 120, 190, 210)
    
    return g

grid = make_complex_grid()

# ==========================================
# Step 1: 基础骨架与距离图
# ==========================================
free = (grid == 255).astype(np.uint8)
dist = ndimage.distance_transform_edt(free)
skeleton = skeletonize(free).astype(np.uint8)

skel_y, skel_x = np.where(skeleton > 0)
skel_set = set(zip(skel_x.tolist(), skel_y.tolist()))

# ==========================================
# Step 2: 提取基础特征点 (端点 & 分叉点)
# ==========================================
kernel = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]])
neighbor_count = ndimage.convolve(skeleton, kernel, mode='constant', cval=0)
endpoint_mask = (neighbor_count == 1) & (skeleton > 0)
junction_mask = (neighbor_count >= 3) & (skeleton > 0)

# ==========================================
# Step 3: 通过向量夹角提取拐弯点
# =================================##
def get_neighbors(x, y, s_set):
    neis = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0: continue
            if (x+dx, y+dy) in s_set: neis.append((x+dx, y+dy))
    return neis

def walk_straight(sx, sy, prev_x, prev_y, steps, s_set):
    """沿着骨架往前走N步，遇到分叉或端点停止，返回路径"""
    path = []
    cx, cy = sx, sy
    for _ in range(steps):
        neis = get_neighbors(cx, cy, s_set)
        neis = [n for n in neis if n != (prev_x, prev_y)]
        if len(neis) != 1: break 
        path.append(neis[0])
        prev_x, prev_y = cx, cy
        cx, cy = neis[0]
    return path

bend_points_set = set()
ANGLE_THRESHOLD = 45 # 调整为45度，确保重要拐点被检测
MAX_STRAIGHT_STEPS = 15 # 15步确保能穿透长走廊

for x, y in skel_set:
    neis = get_neighbors(x, y, skel_set)
    if len(neis) != 2: continue # 只处理度数为2的普通线段点
    
    n1, n2 = neis
    # 向两边各探索更多步，确保能穿透长走廊
    path1 = walk_straight(n1[0], n1[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    path2 = walk_straight(n2[0], n2[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    
    if len(path1) > 0 and len(path2) > 0:
        v1 = np.array(path1[-1]) - np.array([x, y])
        v2 = np.array(path2[-1]) - np.array([x, y])
        n1_norm, n2_norm = np.linalg.norm(v1), np.linalg.norm(v2)
        
        if n1_norm > 0 and n2_norm > 0:
            cos_angle = np.dot(v1, v2) / (n1_norm * n2_norm)
            angle = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
            if angle < (180 - ANGLE_THRESHOLD):
                bend_points_set.add((x, y))

# ==========================================
# Step 4: 优化！密集区域采样
# =================================##
def sample_dense_regions(key_points, max_points_per_region=5):
    """对密集区域进行采样，保留首末点"""
    sampled_points = set()
    
    # 将点转换为numpy数组以便处理
    points = np.array(list(key_points))
    if len(points) == 0:
        return sampled_points
    
    # 使用DBSCAN聚类来识别密集区域
    DENSE_CLUSTER_EPS = 8  # 聚类半径（像素）
    DENSE_CLUSTER_MIN_SAMPLES = 3  # 最小样本数
    
    db = DBSCAN(eps=DENSE_CLUSTER_EPS, min_samples=DENSE_CLUSTER_MIN_SAMPLES).fit(points)
    unique_labels = set(db.labels_)
    
    for label in unique_labels:
        if label == -1:  # 噪声点
            continue
            
        cluster_points = points[db.labels_ == label]
        
        # 如果集群点数超过阈值，进行采样
        if len(cluster_points) > max_points_per_region:
            # 计算集群的中心
            center = np.mean(cluster_points, axis=0)
            
            # 计算每个点到中心的距离
            distances = np.linalg.norm(cluster_points - center, axis=1)
            
            # 选择距离中心最近的点作为代表点
            sorted_indices = np.argsort(distances)
            selected_indices = sorted_indices[:max_points_per_region]
            
            # 添加选中的点
            for idx in selected_indices:
                sampled_points.add(tuple(cluster_points[idx]))
        else:
            # 小集群保留所有点
            for point in cluster_points:
                sampled_points.add(tuple(point))
    
    # 添加噪声点（label=-1的点）
    noise_points = points[db.labels_ == -1]
    for point in noise_points:
        sampled_points.add(tuple(point))
        
    return sampled_points

# ==========================================
# Step 5: 合并所有关键节点并计算 Voronoi Span
# =================================##
key_points_set = set()
ep_y, ep_x = np.where(endpoint_mask)
for x, y in zip(ep_x, ep_y): key_points_set.add((x, y))
jp_y, jp_x = np.where(junction_mask)
for x, y in zip(jp_x, jp_y): key_points_set.add((x, y))
key_points_set.update(bend_points_set)

# 对密集区域进行采样
key_points_set = sample_dense_regions(key_points_set, max_points_per_region=5)

def calc_local_span(x, y, s_set, radius=30):
    """提取节点周围局部骨架的X/Y跨度，代表该处的空间范围"""
    local_pts = []
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            if (x+dx, y+dy) in s_set:
                local_pts.append((x+dx, y+dy))
    if not local_pts: return 0.0, 0.0
    pts = np.array(local_pts)
    # X和Y的极差 * 分辨率 = 物理跨度
    span_x = (np.max(pts[:, 0]) - np.min(pts[:, 0])) * res
    span_y = (np.max(pts[:, 1]) - np.min(pts[:, 1])) * res
    return round(float(span_x), 2), round(float(span_y), 2)

nodes = []
node_id_map = {}
idx = 0
for x, y in key_points_set:
    sx, sy = calc_local_span(x, y, skel_set)
    node_id = f"N_{idx}"
    nodes.append({
        "id": node_id,
        "x": round(float(x * res), 2),
        "y": round(float(y * res), 2),
        "span_x": sx, # 局部X方向物理跨度(米)
        "span_y": sy  # 局部Y方向物理跨度(米)
    })
    node_id_map[(x, y)] = node_id
    idx += 1

print(f"提取到 {len(nodes)} 个纯导航节点 (含拐点)")

# ==========================================
# Step 6: 构建边 (切断节点后的连通域连线)
# =================================##
cut_skeleton = skeleton.copy()
for x, y in key_points_set:
    cut_skeleton[y, x] = 0 # 把节点从骨架上挖空，剩下纯粹的直线段

labels, num = ndimage.label(cut_skeleton > 0)
edges_list = set()

if num > 0:
    # 建立剩余骨架像素到连通域ID的映射
    skel_label_map = {}
    ly, lx = np.where(labels > 0)
    for y, x in zip(ly, lx):
        skel_label_map[(x, y)] = labels[y, x]
        
    # 寻找每个节点连接了哪些直线段(连通域)
    label_to_nodes = {}
    for x, y in key_points_set:
        for nx, ny in get_neighbors(x, y, skel_set):
            if (nx, ny) in skel_label_map:
                l = skel_label_map[(nx, ny)]
                if l not in label_to_nodes: label_to_nodes[l] = []
                label_to_nodes[l].append(node_id_map[(x, y)])
                
    # 同一直线段两端的关键点互相连线
    for l, connected_node_ids in label_to_nodes.items():
        connected_node_ids = list(set(connected_node_ids))
        if len(connected_node_ids) == 2:
            edge = tuple(sorted(connected_node_ids))
            edges_list.add(edge)
        # 理论上切断后，一段直线只能连接2个节点

edges_json = [{"from": e[0], "to": e[1]} for e in edges_list]
print(f"提取到 {len(edges_json)} 条直线路径边")

# ==========================================
# Step 7: 可视化 - 修复方块大小问题
# =================================##
node_dict = {n["id"]: n for n in nodes}

fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(grid, cmap='gray', origin='upper', alpha=0.5)
ax.scatter(skel_x, skel_y, s=0.5, color='lightgray', alpha=0.6)

for n in nodes:
    px, py = n['x'] / res, n['y'] / res
    # 计算方块大小，基于span_x和span_y
    # 使用span_x作为宽度，span_y作为高度，转换为像素大小
    width_px = max(4, min(n['span_x'] * 10, 25))  # 限制在4-25像素之间
    height_px = max(4, min(n['span_y'] * 10, 25)) # 限制在4-25像素之间
    
    # 画节点，大小反映span信息
    ax.plot(px, py, 's', markersize=width_px, markeredgewidth=1.5, 
            color='red', markeredgecolor='black')
    
    # 标注 span
    ax.annotate(f"{n['span_x']}x{n['span_y']}", (px, py),
                textcoords="offset points", xytext=(8, 8), fontsize=7, color='darkred')

for e in edges_json:
    n1, n2 = node_dict[e["from"]], node_dict[e["to"]]
    ax.plot([n1['x']/res, n2['x']/res], [n1['y']/res, n2['y']/res], 'g-', linewidth=2, alpha=0.8)

ax.set_title("Optimized Complex Map Voronoi Span Topology\n(Red Square=Node, Size=Local Span, Green=Straight Path)", fontsize=13)
ax.set_xlim(0, W); ax.set_ylim(H, 0)
plt.tight_layout()
plt.savefig('optimized_complex_map_topology.png', dpi=150, bbox_inches='tight')
plt.close()

# ==========================================
# Step 8: 输出 LLM JSON
# =================================##
print("\n" + "=" * 50)
print("JSON for LLM:")
print("=" * 50)
print(json.dumps({"nodes": nodes, "edges": edges_json}, indent=2, ensure_ascii=False))


提取到 95 个纯导航节点 (含拐点)
提取到 16 条直线路径边

JSON for LLM:
{
  "nodes": [
    {
      "id": "N_0",
      "x": 18.7,
      "y": 3.9,
      "span_x": 3.0,
      "span_y": 6.0
    },
    {
      "id": "N_1",
      "x": 17.9,
      "y": 9.0,
      "span_x": 2.2,
      "span_y": 6.0
    },
    {
      "id": "N_2",
      "x": 13.4,
      "y": 6.7,
      "span_x": 3.0,
      "span_y": 6.0
    },
    {
      "id": "N_3",
      "x": 17.8,
      "y": 10.9,
      "span_x": 2.5,
      "span_y": 6.0
    },
    {
      "id": "N_4",
      "x": 17.9,
      "y": 10.8,
      "span_x": 2.5,
      "span_y": 6.0
    },
    {
      "id": "N_5",
      "x": 16.0,
      "y": 1.2,
      "span_x": 6.0,
      "span_y": 3.6
    },
    {
      "id": "N_6",
      "x": 6.5,
      "y": 7.0,
      "span_x": 3.0,
      "span_y": 3.9
    },
    {
      "id": "N_7",
      "x": 5.2,
      "y": 1.5,
      "span_x": 6.0,
      "span_y": 3.9
    },
    {
      "id": "N_8",
      "x": 6.9,
      "y": 4.0,
      "span_x": 6.0,
      "spa

In [30]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage.morphology import skeletonize
import json
from sklearn.cluster import DBSCAN

W, H, res = 200, 150, 0.1

def make_complex_grid():
    g = np.ones((H, W), dtype=np.uint8) * 255
    def w(y1, y2, x1, x2): g[y1:y2, x1:x2] = 0
    def c(cy, cx, r):
        Y, X = np.ogrid[:H, :W]
        g[(X - cx)**2 + (Y - cy)**2 <= r**2] = 0
    
    # 外墙
    w(0, 3, 0, W); w(H-3, H, 0, W); w(0, H, 0, 3); w(0, H, W-3, W)
    
    # 主走廊
    w(30, 120, 40, 60)
    w(30, 120, 140, 160)
    
    # 左侧房间
    w(10, 30, 10, 40)
    w(10, 30, 70, 100)
    w(10, 30, 120, 150)
    
    # 右侧房间
    w(10, 30, 170, 200)
    w(10, 30, 220, 250)
    
    # 中间房间
    w(50, 70, 80, 120)
    w(50, 70, 180, 220)
    
    # 上方房间
    w(80, 100, 50, 90)
    w(80, 100, 130, 170)
    w(80, 100, 210, 250)
    
    # 下方房间
    w(120, 140, 70, 110)
    w(120, 140, 150, 190)
    w(120, 140, 230, 270)
    
    # 添加一些圆形障碍物
    c(25, 25, 3)
    c(25, 175, 3)
    c(125, 25, 3)
    c(125, 175, 3)
    c(75, 75, 4)
    c(175, 75, 4)
    c(75, 125, 4)
    c(175, 125, 4)
    
    # 添加一些不规则形状
    w(60, 80, 30, 50)
    w(60, 80, 190, 210)
    w(100, 120, 30, 50)
    w(100, 120, 190, 210)
    
    return g

grid = make_complex_grid()

# ==========================================
# Step 1: 基础骨架与距离图
# ==========================================
free = (grid == 255).astype(np.uint8)
dist = ndimage.distance_transform_edt(free)
skeleton = skeletonize(free).astype(np.uint8)

skel_y, skel_x = np.where(skeleton > 0)
skel_set = set(zip(skel_x.tolist(), skel_y.tolist()))

# ==========================================
# Step 2: 提取基础特征点 (端点 & 分叉点)
# ==========================================
kernel = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]])
neighbor_count = ndimage.convolve(skeleton, kernel, mode='constant', cval=0)
endpoint_mask = (neighbor_count == 1) & (skeleton > 0)
junction_mask = (neighbor_count >= 3) & (skeleton > 0)

# ==========================================
# Step 3: 通过向量夹角提取拐弯点
# =================================##
def get_neighbors(x, y, s_set):
    neis = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0: continue
            if (x+dx, y+dy) in s_set: neis.append((x+dx, y+dy))
    return neis

def walk_straight(sx, sy, prev_x, prev_y, steps, s_set):
    """沿着骨架往前走N步，遇到分叉或端点停止，返回路径"""
    path = []
    cx, cy = sx, sy
    for _ in range(steps):
        neis = get_neighbors(cx, cy, s_set)
        neis = [n for n in neis if n != (prev_x, prev_y)]
        if len(neis) != 1: break 
        path.append(neis[0])
        prev_x, prev_y = cx, cy
        cx, cy = neis[0]
    return path

bend_points_set = set()
ANGLE_THRESHOLD = 45 # 调整为45度，确保重要拐点被检测
MAX_STRAIGHT_STEPS = 15 # 15步确保能穿透长走廊

for x, y in skel_set:
    neis = get_neighbors(x, y, skel_set)
    if len(neis) != 2: continue # 只处理度数为2的普通线段点
    
    n1, n2 = neis
    # 向两边各探索更多步，确保能穿透长走廊
    path1 = walk_straight(n1[0], n1[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    path2 = walk_straight(n2[0], n2[1], x, y, MAX_STRAIGHT_STEPS, skel_set)
    
    if len(path1) > 0 and len(path2) > 0:
        v1 = np.array(path1[-1]) - np.array([x, y])
        v2 = np.array(path2[-1]) - np.array([x, y])
        n1_norm, n2_norm = np.linalg.norm(v1), np.linalg.norm(v2)
        
        if n1_norm > 0 and n2_norm > 0:
            cos_angle = np.dot(v1, v2) / (n1_norm * n2_norm)
            angle = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
            if angle < (180 - ANGLE_THRESHOLD):
                bend_points_set.add((x, y))

# ==========================================
# Step 4: 优化！密集区域采样
# =================================##
def sample_dense_regions(key_points, max_points_per_region=5):
    """对密集区域进行采样，保留首末点"""
    sampled_points = set()
    
    # 将点转换为numpy数组以便处理
    points = np.array(list(key_points))
    if len(points) == 0:
        return sampled_points
    
    # 使用DBSCAN聚类来识别密集区域
    DENSE_CLUSTER_EPS = 8  # 聚类半径（像素）
    DENSE_CLUSTER_MIN_SAMPLES = 3  # 最小样本数
    
    db = DBSCAN(eps=DENSE_CLUSTER_EPS, min_samples=DENSE_CLUSTER_MIN_SAMPLES).fit(points)
    unique_labels = set(db.labels_)
    
    for label in unique_labels:
        if label == -1:  # 噪声点
            continue
            
        cluster_points = points[db.labels_ == label]
        
        # 如果集群点数超过阈值，进行采样
        if len(cluster_points) > max_points_per_region:
            # 计算集群的中心
            center = np.mean(cluster_points, axis=0)
            
            # 计算每个点到中心的距离
            distances = np.linalg.norm(cluster_points - center, axis=1)
            
            # 选择距离中心最近的点作为代表点
            sorted_indices = np.argsort(distances)
            selected_indices = sorted_indices[:max_points_per_region]
            
            # 添加选中的点
            for idx in selected_indices:
                sampled_points.add(tuple(cluster_points[idx]))
        else:
            # 小集群保留所有点
            for point in cluster_points:
                sampled_points.add(tuple(point))
    
    # 添加噪声点（label=-1的点）
    noise_points = points[db.labels_ == -1]
    for point in noise_points:
        sampled_points.add(tuple(point))
        
    return sampled_points

# ==========================================
# Step 5: 合并所有关键节点并计算 Voronoi Span
# =================================##
key_points_set = set()
ep_y, ep_x = np.where(endpoint_mask)
for x, y in zip(ep_x, ep_y): key_points_set.add((x, y))
jp_y, jp_x = np.where(junction_mask)
for x, y in zip(jp_x, jp_y): key_points_set.add((x, y))
key_points_set.update(bend_points_set)

# 对密集区域进行采样
key_points_set = sample_dense_regions(key_points_set, max_points_per_region=5)

def calc_local_span(x, y, s_set, radius=30):
    """提取节点周围局部骨架的X/Y跨度，代表该处的空间范围"""
    local_pts = []
    for dx in range(-radius, radius+1):
        for dy in range(-radius, radius+1):
            if (x+dx, y+dy) in s_set:
                local_pts.append((x+dx, y+dy))
    if not local_pts: return 0.0, 0.0
    pts = np.array(local_pts)
    # X和Y的极差 * 分辨率 = 物理跨度
    span_x = (np.max(pts[:, 0]) - np.min(pts[:, 0])) * res
    span_y = (np.max(pts[:, 1]) - np.min(pts[:, 1])) * res
    return round(float(span_x), 2), round(float(span_y), 2)

nodes = []
node_id_map = {}
idx = 0
for x, y in key_points_set:
    sx, sy = calc_local_span(x, y, skel_set)
    node_id = f"N_{idx}"
    nodes.append({
        "id": node_id,
        "x": round(float(x * res), 2),
        "y": round(float(y * res), 2),
        "span_x": sx, # 局部X方向物理跨度(米)
        "span_y": sy  # 局部Y方向物理跨度(米)
    })
    node_id_map[(x, y)] = node_id
    idx += 1

print(f"提取到 {len(nodes)} 个纯导航节点 (含拐点)")

# ==========================================
# Step 6: 重新设计！基于骨架追踪的边计算
# =================================##
def trace_path_between_nodes(start_node, end_node, s_set, max_steps=100):
    """在两个节点之间追踪骨架路径"""
    # 获取节点的像素坐标
    start_x, start_y = start_node
    end_x, end_y = end_node
    
    # 从起点开始追踪
    path = [(start_x, start_y)]
    current_x, current_y = start_x, start_y
    prev_x, prev_y = -1, -1
    
    for _ in range(max_steps):
        # 找到当前点的邻居
        neighbors = get_neighbors(current_x, current_y, s_set)
        # 排除已经走过的点
        neighbors = [n for n in neighbors if n != (prev_x, prev_y)]
        
        if len(neighbors) == 0:
            break
            
        # 如果到达终点，停止追踪
        if (current_x, current_y) == (end_x, end_y):
            break
            
        # 选择下一个点（简单策略：选择第一个邻居）
        next_x, next_y = neighbors[0]
        path.append((next_x, next_y))
        
        # 更新当前位置
        prev_x, prev_y = current_x, current_y
        current_x, current_y = next_x, next_y
        
        # 如果到达终点，停止追踪
        if (current_x, current_y) == (end_x, end_y):
            break
    
    return path

# 创建节点到像素坐标的映射
node_pixel_map = {n["id"]: (int(n["x"]/res), int(n["y"]/res)) for n in nodes}

# 计算边
edges_list = set()
node_ids = list(node_id_map.values())

# 为每个节点找到所有直接连接的节点
for i, node_id in enumerate(node_ids):
    node_x, node_y = node_pixel_map[node_id]
    
    # 找到该节点的邻居（在骨架上直接相邻的节点）
    neighbors = []
    for j, other_node_id in enumerate(node_ids):
        if i == j: continue
        
        other_x, other_y = node_pixel_map[other_node_id]
        
        # 检查两个节点是否在骨架上直接相邻
        if (other_x, other_y) in get_neighbors(node_x, node_y, skel_set):
            neighbors.append(other_node_id)
    
    # 为每个邻居创建边
    for neighbor_id in neighbors:
        edge = tuple(sorted([node_id, neighbor_id]))
        edges_list.add(edge)

edges_json = [{"from": e[0], "to": e[1]} for e in edges_list]
print(f"提取到 {len(edges_json)} 条直线路径边")

# ==========================================
# Step 7: 可视化
# =================================##
node_dict = {n["id"]: n for n in nodes}

fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(grid, cmap='gray', origin='upper', alpha=0.5)
ax.scatter(skel_x, skel_y, s=0.5, color='lightgray', alpha=0.6)

for n in nodes:
    px, py = n['x'] / res, n['y'] / res
    # 计算方块大小，基于span_x和span_y
    width_px = max(4, min(n['span_x'] * 10, 25))  # 限制在4-25像素之间
    height_px = max(4, min(n['span_y'] * 10, 25)) # 限制在4-25像素之间
    
    # 画节点，大小反映span信息
    ax.plot(px, py, 's', markersize=width_px, markeredgewidth=1.5, 
            color='red', markeredgecolor='black')
    
    # 标注 span
    ax.annotate(f"{n['span_x']}x{n['span_y']}", (px, py),
                textcoords="offset points", xytext=(8, 8), fontsize=7, color='darkred')

for e in edges_json:
    n1, n2 = node_dict[e["from"]], node_dict[e["to"]]
    ax.plot([n1['x']/res, n2['x']/res], [n1['y']/res, n2['y']/res], 'g-', linewidth=2, alpha=0.8)

ax.set_title("Optimized Complex Map Voronoi Span Topology\n(Red Square=Node, Size=Local Span, Green=Straight Path)", fontsize=13)
ax.set_xlim(0, W); ax.set_ylim(H, 0)
plt.tight_layout()
plt.savefig('optimized_complex_map_topology.png', dpi=150, bbox_inches='tight')
plt.close()

# ==========================================
# Step 8: 输出 LLM JSON
# =================================##
print("\n" + "=" * 50)
print("JSON for LLM:")
print("=" * 50)
print(json.dumps({"nodes": nodes, "edges": edges_json}, indent=2, ensure_ascii=False))


提取到 95 个纯导航节点 (含拐点)
提取到 43 条直线路径边

JSON for LLM:
{
  "nodes": [
    {
      "id": "N_0",
      "x": 18.7,
      "y": 3.9,
      "span_x": 3.0,
      "span_y": 6.0
    },
    {
      "id": "N_1",
      "x": 17.9,
      "y": 9.0,
      "span_x": 2.2,
      "span_y": 6.0
    },
    {
      "id": "N_2",
      "x": 13.4,
      "y": 6.7,
      "span_x": 3.0,
      "span_y": 6.0
    },
    {
      "id": "N_3",
      "x": 17.8,
      "y": 10.9,
      "span_x": 2.5,
      "span_y": 6.0
    },
    {
      "id": "N_4",
      "x": 17.9,
      "y": 10.8,
      "span_x": 2.5,
      "span_y": 6.0
    },
    {
      "id": "N_5",
      "x": 16.0,
      "y": 1.2,
      "span_x": 6.0,
      "span_y": 3.6
    },
    {
      "id": "N_6",
      "x": 6.5,
      "y": 7.0,
      "span_x": 3.0,
      "span_y": 3.9
    },
    {
      "id": "N_7",
      "x": 5.2,
      "y": 1.5,
      "span_x": 6.0,
      "span_y": 3.9
    },
    {
      "id": "N_8",
      "x": 6.9,
      "y": 4.0,
      "span_x": 6.0,
      "spa